In [ ]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import os
import time

# -----------------------------
# 1️⃣ 设置股票列表与时间
# -----------------------------
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
start_date = '2013-01-01'
end_date = '2023-12-31'

# 临时保存目录
temp_dir = "stock_temp"
os.makedirs(temp_dir, exist_ok=True)

# -----------------------------
# 2️⃣ 下载每支股票（断点续传）
# -----------------------------
for ticker in tickers:
    temp_file = os.path.join(temp_dir, f"{ticker}.csv")
    
    if os.path.exists(temp_file):
        print(f"{ticker} 已下载过，跳过")
        continue
    
    attempt = 0
    while attempt < 10:
        try:
            print(f"下载 {ticker} ...")
            df = yf.download(ticker, start=start_date, end=end_date)
            if df.empty:
                raise ValueError(f"{ticker} 数据为空")
            
            df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
            df.to_csv(temp_file)
            print(f"{ticker} 下载完成")
            time.sleep(5)  # 每支股票间隔，降低限速风险
            break
        except Exception as e:
            attempt += 1
            wait_time = 120 * attempt  # 指数退避
            print(f"{ticker} 下载失败或限速，等待 {wait_time} 秒重试... 错误: {e}")
            time.sleep(wait_time)

# -----------------------------
# 3️⃣ 合并所有股票数据
# -----------------------------
all_data = []
valid_tickers = []

for ticker in tickers:
    temp_file = os.path.join(temp_dir, f"{ticker}.csv")
    if not os.path.exists(temp_file):
        print(f"{ticker} 数据缺失，跳过")
        continue
    
    df = pd.read_csv(temp_file, index_col=0)
    df.columns = [f"{ticker}_{c}" for c in df.columns]
    all_data.append(df)
    valid_tickers.append(ticker)

if not all_data:
    raise ValueError("没有成功下载任何股票数据")

data = pd.concat(all_data, axis=1).dropna()
print(f"合并完成，数据维度: {data.shape}")

# -----------------------------
# 4️⃣ 计算技术指标
# -----------------------------
for ticker in valid_tickers:
    close = data[f'{ticker}_Close']
    high = data[f'{ticker}_High']
    low = data[f'{ticker}_Low']
    vol = data[f'{ticker}_Volume']

    data[f'{ticker}_SMA10'] = ta.sma(close, length=10)
    data[f'{ticker}_EMA10'] = ta.ema(close, length=10)
    data[f'{ticker}_RSI14'] = ta.rsi(close, length=14)
    macd = ta.macd(close)
    data[f'{ticker}_MACD'] = macd['MACD_12_26_9']
    data[f'{ticker}_MACD_signal'] = macd['MACDs_12_26_9']
    bb = ta.bbands(close)
    data[f'{ticker}_BBU'] = bb['BBU_20_2.0']
    data[f'{ticker}_BBL'] = bb['BBL_20_2.0']
    atr = ta.atr(high, low, close)
    data[f'{ticker}_ATR14'] = atr

# -----------------------------
# 5️⃣ 多支股票交叉特征
# -----------------------------
for i in range(len(valid_tickers)):
    for j in range(i+1, len(valid_tickers)):
        t1, t2 = valid_tickers[i], valid_tickers[j]
        data[f'{t1}_{t2}_ratio'] = data[f'{t1}_Close'] / data[f'{t2}_Close']
        data[f'{t1}_{t2}_diff'] = data[f'{t1}_Close'] - data[f'{t2}_Close']
        data[f'{t1}_{t2}_corr20'] = data[f'{t1}_Close'].rolling(20).corr(data[f'{t2}_Close'])

# -----------------------------
# 6️⃣ 去掉缺失值
# -----------------------------
data = data.dropna()
print(f"最终数据维度: {data.shape}")

# -----------------------------
# 7️⃣ 保存为 CSV
# -----------------------------
output_file = "multi_stock_features.csv"
data.to_csv(output_file)
print(f"数据已保存为: {output_file}")